In [1]:
# ============================================================
# CELL 0: Surgically patch datasets/features/audio.py on disk
# to remove the broken torchcodec import.
# Run this FIRST before any other imports.
# ============================================================

import datasets.features.audio as _a
import inspect, pathlib, re

audio_path = pathlib.Path(inspect.getfile(_a))
print(f"Patching: {audio_path}")

original = audio_path.read_text(encoding="utf-8")

# ── Patch 1: neutralise the torchcodec import block ──
# The file has something like:
#   try:
#       from datasets.features import _torchcodec
#       ...
#   except ...:
#       ...
patched = re.sub(
    r'try\s*:\s*\n(\s+from datasets\.features import _torchcodec.*?)\nexcept[^\n]*:\s*\n[^\n]*\n',
    '# torchcodec import removed by patch\n',
    original,
    flags=re.DOTALL,
)

# ── Patch 2: replace the torchcodec decode branch inside decode_example ──
# Wherever _torchcodec.decode(...) is called, replace with a soundfile call
patched = re.sub(
    r'_torchcodec\.decode\([^)]*\)',
    'None  # torchcodec removed',
    patched,
)

# ── Patch 3: ensure the sf.read branch is always taken ──
# The file guards the sf.read call with:
#   if _torchcodec is not None:   <torchcodec path>
#   else:                          sf.read path
# Make the condition always false so sf.read runs every time.
patched = re.sub(
    r'if _torchcodec is not None:',
    'if False:  # torchcodec disabled by patch',
    patched,
)

if patched == original:
    print("⚠️  Regex patterns didn't match — falling back to brute-force replace.")
    # Brute-force: comment out every line that mentions _torchcodec
    lines = original.splitlines()
    new_lines = []
    for line in lines:
        if "_torchcodec" in line:
            new_lines.append("# PATCHED: " + line)
        else:
            new_lines.append(line)
    patched = "\n".join(new_lines)

audio_path.write_text(patched, encoding="utf-8")
print("✅ audio.py patched on disk.")
print("⚠️  Now go to Runtime → Restart session, then run Cell 1 onwards.")
print("   Do NOT re-run this cell after restarting.")

Patching: /usr/local/lib/python3.12/dist-packages/datasets/features/audio.py
⚠️  Regex patterns didn't match — falling back to brute-force replace.
✅ audio.py patched on disk.
⚠️  Now go to Runtime → Restart session, then run Cell 1 onwards.
   Do NOT re-run this cell after restarting.


In [1]:
import subprocess

def run(cmd):
    subprocess.run(cmd, shell=True, check=False)

# Core torch
run("pip install -q torch==2.6.0 torchaudio==2.6.0 torchvision==0.21.0")

# HF ecosystem
run("pip install -q transformers==4.41.2 huggingface-hub==0.24.7 datasets==2.20.0")

# Audio
run("pip install -q soundfile librosa")
run("apt-get install -qq -y ffmpeg")

# CLS tokenizer stack for sooktam2
run("pip install -q indic-nlp-library==0.92 indic-unified-parser==1.0.6 indo-arabic-transliteration==0.1.5 indic-numtowords==1.1.0")
run("pip install -q urduhack==1.1.1")
run("pip install -q git+https://github.com/libindic/indic-trans.git@0287fa62289968f0ce06cbe2df61cfadf4088c75")
run("pip install -q keras==2.15.0 tensorflow==2.15.0 tensorflow-addons==0.23.0")

# F5-TTS deps
run("pip install -q ema-pytorch==0.7.9 x-transformers vocos torchdiffeq==0.2.5 cached-path")
run("pip install -q safetensors accelerate click==8.0.1 tqdm numpy pydub")

# Clone sooktam2 (skip if already cloned)
import os
if not os.path.exists("/content/sooktam2"):
    run("git clone https://huggingface.co/bharatgenai/sooktam2 /content/sooktam2")
run("pip install -e /content/sooktam2 --no-cache-dir -q")

print("✅ All dependencies installed.")

✅ All dependencies installed.


In [2]:
# Quick check — this import must NOT raise ModuleNotFoundError
import datasets.features.audio as _a
import inspect

src = inspect.getsource(_a)
if "_torchcodec" not in src or "PATCHED" in src or "torchcodec disabled" in src:
    print("✅ Patch is active — torchcodec is neutralised.")
else:
    # The pip install in Cell 1 may have overwritten our patched file.
    # Re-apply the patch right now without needing another restart,
    # because we haven't iterated over the dataset yet.
    import pathlib, re
    audio_path = pathlib.Path(inspect.getfile(_a))
    original   = audio_path.read_text(encoding="utf-8")
    lines      = original.splitlines()
    new_lines  = []
    for line in lines:
        if "_torchcodec" in line:
            new_lines.append("# PATCHED: " + line)
        else:
            new_lines.append(line)
    patched = "\n".join(new_lines)
    audio_path.write_text(patched, encoding="utf-8")

    # Force Python to reload the module from the new file
    import importlib
    importlib.reload(_a)

    # Also patch the live Audio class's decode_example to use soundfile
    import io, numpy as np, soundfile as sf

    _orig_decode = _a.Audio.decode_example

    def _safe_decode(self, value, token_per_repo_id=None):
        if not self.decode:
            return value
        path  = value.get("path")
        raw   = value.get("bytes")
        arr   = value.get("array")
        sr    = value.get("sampling_rate")
        if arr is not None:
            arr = np.array(arr, dtype=np.float32)
            if arr.ndim > 1: arr = arr.mean(axis=1)
            return {"path": path, "array": arr, "sampling_rate": sr}
        buf = io.BytesIO(raw) if raw else (open(path,"rb") if path else None)
        if buf is None:
            return value
        arr, sr = sf.read(buf, dtype="float32", always_2d=False)
        if arr.ndim > 1: arr = arr.mean(axis=1)
        target = self.sampling_rate or sr
        if target != sr:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=target)
            sr  = target
        return {"path": path, "array": arr, "sampling_rate": sr}

    import types
    _a.Audio.decode_example = types.MethodType(
        lambda self, value, token_per_repo_id=None: _safe_decode(self, value, token_per_repo_id),
        _a.Audio
    )
    # Patch it as an unbound method instead
    _a.Audio.decode_example = _safe_decode
    print("✅ Re-patched live Audio.decode_example → soundfile only.")

✅ Patch is active — torchcodec is neutralised.


In [3]:
# CELL 2: Login to HuggingFace

from huggingface_hub import login
login()
print("Logged in to HuggingFace.")

Logged in to HuggingFace.


In [4]:
# ============================================================
# CELL 3: Stream IndicVoices_R Hindi, pick 1 male + 1 female
# speaker with enough samples, then build train/test split.
# We do NOT download the whole dataset — we stream and collect
# only what we need.
# ============================================================

import os, json, random
import soundfile as sf
import numpy as np
from datasets import load_dataset

random.seed(42)
np.random.seed(42)

LANG_CONFIG = "Hindi"          # config name in indicvoices_r
TARGET_GENDER_M = "male"
TARGET_GENDER_F = "female"
MIN_SAMPLES_PER_SPEAKER = 15   # need train + test utterances
TRAIN_SIZE = 10
TEST_SIZE = 5                  # unseen test utterances per speaker

os.makedirs("/content/indicvoices_hindi", exist_ok=True)
os.makedirs("/content/indicvoices_hindi/male", exist_ok=True)
os.makedirs("/content/indicvoices_hindi/female", exist_ok=True)

print(f"Streaming ai4bharat/indicvoices_r [{LANG_CONFIG}] train split...")
ds = load_dataset(
    "ai4bharat/indicvoices_r",
    LANG_CONFIG,
    split="train",
    streaming=True,
    trust_remote_code=True,
)

# Collect samples per speaker, keeping only male & female
speaker_buckets = {}   # speaker_id -> list of samples

MAX_SCAN = 20000   # scan this many rows to find good speakers

for i, sample in enumerate(ds):
    if i >= MAX_SCAN:
        break

    gender  = sample.get("gender", "").strip().lower()
    spk_id  = sample.get("speaker_id", "").strip()
    text    = (sample.get("normalized") or sample.get("text") or "").strip()
    audio   = sample.get("audio")

    if gender not in (TARGET_GENDER_M, TARGET_GENDER_F):
        continue
    if not spk_id or not text or audio is None:
        continue

    if spk_id not in speaker_buckets:
        speaker_buckets[spk_id] = {"gender": gender, "samples": []}

    speaker_buckets[spk_id]["samples"].append({
        "text": text,
        "audio_array": audio["array"],
        "sampling_rate": audio["sampling_rate"],
    })

    # Early exit once we have enough for both genders
    male_done   = any(v["gender"] == TARGET_GENDER_M and len(v["samples"]) >= MIN_SAMPLES_PER_SPEAKER
                      for v in speaker_buckets.values())
    female_done = any(v["gender"] == TARGET_GENDER_F and len(v["samples"]) >= MIN_SAMPLES_PER_SPEAKER
                      for v in speaker_buckets.values())
    if male_done and female_done:
        print(f"  Found sufficient speakers after scanning {i+1} rows.")
        break

# Pick the speaker with most samples for each gender
def pick_best_speaker(buckets, gender, min_samples):
    candidates = [(sid, v) for sid, v in buckets.items()
                  if v["gender"] == gender and len(v["samples"]) >= min_samples]
    if not candidates:
        return None, None
    candidates.sort(key=lambda x: len(x[1]["samples"]), reverse=True)
    return candidates[0]

male_spk_id,   male_data   = pick_best_speaker(speaker_buckets, TARGET_GENDER_M, MIN_SAMPLES_PER_SPEAKER)
female_spk_id, female_data = pick_best_speaker(speaker_buckets, TARGET_GENDER_F, MIN_SAMPLES_PER_SPEAKER)

assert male_spk_id,   "❌ No male speaker found with enough samples. Increase MAX_SCAN."
assert female_spk_id, "❌ No female speaker found with enough samples. Increase MAX_SCAN."

print(f"\n✅ Selected male speaker  : {male_spk_id}  ({len(male_data['samples'])} samples)")
print(f"✅ Selected female speaker: {female_spk_id}  ({len(female_data['samples'])} samples)")

# ---------- Build train / test splits ----------
def split_speaker(data, train_size, test_size):
    samples = data["samples"][:]
    random.shuffle(samples)
    train = samples[:train_size]
    test  = samples[train_size : train_size + test_size]
    return train, test

male_train,   male_test   = split_speaker(male_data,   TRAIN_SIZE, TEST_SIZE)
female_train, female_test = split_speaker(female_data, TRAIN_SIZE, TEST_SIZE)

print(f"\nMale   — train: {len(male_train)}, test: {len(male_test)}")
print(f"Female — train: {len(female_train)}, test: {len(female_test)}")

# ---------- Save audio to disk ----------
def save_speaker_samples(samples, gender, split_name, spk_id):
    saved = []
    base  = f"/content/indicvoices_hindi/{gender}/{split_name}"
    os.makedirs(base, exist_ok=True)
    for idx, s in enumerate(samples):
        arr = np.array(s["audio_array"], dtype=np.float32)
        sr  = s["sampling_rate"]
        wav_path = os.path.join(base, f"{spk_id}_{split_name}_{idx:03d}.wav")
        sf.write(wav_path, arr, sr)
        saved.append({"wav_path": wav_path, "text": s["text"], "speaker_id": spk_id, "sr": sr})
    return saved

male_train_saved   = save_speaker_samples(male_train,   "male",   "train", male_spk_id)
male_test_saved    = save_speaker_samples(male_test,    "male",   "test",  male_spk_id)
female_train_saved = save_speaker_samples(female_train, "female", "train", female_spk_id)
female_test_saved  = save_speaker_samples(female_test,  "female", "test",  female_spk_id)

# Save manifest JSON for reference
manifest = {
    "male_speaker_id":   male_spk_id,
    "female_speaker_id": female_spk_id,
    "male_train":   male_train_saved,
    "male_test":    male_test_saved,
    "female_train": female_train_saved,
    "female_test":  female_test_saved,
}
with open("/content/indicvoices_hindi/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\n✅ Train/test splits saved to /content/indicvoices_hindi/")
print("   manifest.json written.")

Streaming ai4bharat/indicvoices_r [Hindi] train split...


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/99 [00:00<?, ?it/s]

  Found sufficient speakers after scanning 256 rows.

✅ Selected male speaker  : S4259199200352358  (18 samples)
✅ Selected female speaker: S4256901000332608  (15 samples)

Male   — train: 10, test: 5
Female — train: 10, test: 5

✅ Train/test splits saved to /content/indicvoices_hindi/
   manifest.json written.


In [5]:
# ============================================================
# CELL 4: Pick the reference audio for each gender.
# We choose the train sample whose text is longest (most
# phonetically rich), which gives F5-TTS the best voice
# conditioning signal. 3–10 sec is ideal.
# ============================================================

import librosa

def pick_reference(train_saved, target_duration=(3, 10)):
    """Pick sample closest to 6 seconds from the training set."""
    scored = []
    for s in train_saved:
        y, sr = librosa.load(s["wav_path"], sr=None)
        dur = len(y) / sr
        score = abs(dur - 6.0)          # prefer ~6 sec
        scored.append((score, dur, s))
    scored.sort(key=lambda x: x[0])
    _, dur, best = scored[0]
    return best, dur

male_ref, male_ref_dur     = pick_reference(male_train_saved)
female_ref, female_ref_dur = pick_reference(female_train_saved)

print(f"Male   reference : {male_ref['wav_path']}")
print(f"  text : {male_ref['text']}")
print(f"  dur  : {male_ref_dur:.2f}s\n")

print(f"Female reference : {female_ref['wav_path']}")
print(f"  text : {female_ref['text']}")
print(f"  dur  : {female_ref_dur:.2f}s")

# Persist for downstream cells
MALE_REF_WAV   = male_ref["wav_path"]
MALE_REF_TEXT  = male_ref["text"]
FEMALE_REF_WAV = female_ref["wav_path"]
FEMALE_REF_TEXT = female_ref["text"]

Male   reference : /content/indicvoices_hindi/male/train/S4259199200352358_train_002.wav
  text : भारत के बहोत ही बड़े इंडस्ट्रायल मुकेश अंबानी इनका जिओ का सिम है
  dur  : 7.55s

Female reference : /content/indicvoices_hindi/female/train/S4256901000332608_train_007.wav
  text : करेंगे कि ऐसे नहीं ऐसे जाओ यहाँ से यहाँ पर इलाज नहीं हो पाएगा इमरजेंसी है
  dur  : 5.80s


In [6]:
# ============================================================
# CELL 5: Install f5_tts properly, then load sooktam2
# ============================================================

import subprocess, sys, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout[-2000:])
    if result.stderr: print(result.stderr[-1000:])
    return result.returncode

# ── Step 1: Install f5-tts from PyPI (the canonical package) ──
print("Installing f5-tts from PyPI...")
run("pip install -q f5-tts")

# ── Step 2: Verify f5_tts is importable ──
try:
    import f5_tts
    print(f"✅ f5_tts found at: {f5_tts.__file__}")
except ImportError as e:
    print(f"PyPI f5-tts failed ({e}), trying from source...")
    # Fallback: install directly from F5-TTS GitHub
    run("pip install -q git+https://github.com/SWivid/F5-TTS.git")
    try:
        import f5_tts
        print(f"✅ f5_tts (from source) at: {f5_tts.__file__}")
    except ImportError as e2:
        print(f"❌ Still failing: {e2}")
        sys.exit(1)

# ── Step 3: Re-install sooktam2 editable after f5_tts is confirmed ──
if os.path.exists("/content/sooktam2"):
    print("\nRe-installing sooktam2 editable package...")
    run("pip install -e /content/sooktam2 --no-cache-dir -q")
else:
    print("Cloning sooktam2...")
    run("git clone https://huggingface.co/bharatgenai/sooktam2 /content/sooktam2")
    run("pip install -e /content/sooktam2 --no-cache-dir -q")

# ── Step 4: Confirm sooktam2's own modules are importable ──
sys.path.insert(0, "/content/sooktam2/src")   # some repos put src/ in path
sys.path.insert(0, "/content/sooktam2")

# ── Step 5: Load the model ──
from transformers import AutoModel

print("\nLoading bharatgenai/sooktam2 (downloads ~6.7 GB on first run)...")
tts_model = AutoModel.from_pretrained(
    "bharatgenai/sooktam2",
    trust_remote_code=True,
)
print("✅ sooktam2 model loaded successfully.")

Installing f5-tts from PyPI...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 8.1 MB/s eta 0:00:00

ERROR: pip's 

config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

hf_auto.py:   0%|          | 0.00/463 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/bharatgenai/sooktam2:
- hf_auto.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid e

model_1250000.pt:   0%|          | 0.00/5.38G [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Download Vocos from huggingface charactr/vocos-mel-24khz


config.yaml:   0%|          | 0.00/461 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/54.4M [00:00<?, ?B/s]


vocab :  /root/.cache/huggingface/hub/models--bharatgenai--sooktam2/snapshots/eb270c0ceebd58e0e513820eb71c637e9f486a7b/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--bharatgenai--sooktam2/snapshots/eb270c0ceebd58e0e513820eb71c637e9f486a7b/model_1250000.pt 

✅ sooktam2 model loaded successfully.


In [7]:
# ============================================================
# CELL 6: For the TRAIN/TEST split, generate TTS using the
# reference voice and compare with ground-truth audio.
# OUTPUT structure:
#   /content/outputs/
#     male/
#       train/ gt_000.wav  gen_000.wav  ...
#       test/  gt_000.wav  gen_000.wav  ...   ← unseen!
#     female/
#       train/ ...
#       test/  ...
# ============================================================

import os, json
import soundfile as sf

OUTPUT_ROOT = "/content/outputs"

def generate_for_split(split_saved, ref_wav, ref_text, gender, split_name):
    out_dir = os.path.join(OUTPUT_ROOT, gender, split_name)
    os.makedirs(out_dir, exist_ok=True)
    results = []

    for idx, sample in enumerate(split_saved):
        gen_text = sample["text"]
        out_gt   = os.path.join(out_dir, f"gt_{idx:03d}.wav")
        out_gen  = os.path.join(out_dir, f"gen_{idx:03d}.wav")

        # Copy ground-truth
        import shutil
        shutil.copy(sample["wav_path"], out_gt)

        # Generate TTS
        try:
            wav, sr, _ = tts_model.infer(
                ref_file  = ref_wav,
                ref_text  = ref_text,
                gen_text  = gen_text,
                tokenizer = "cls",
                cls_language = "hindi",
                file_wave = out_gen,
            )
            status = "ok"
            print(f"  [{gender}/{split_name}] {idx+1}/{len(split_saved)} ✅ gen saved")
        except Exception as e:
            status = f"error: {e}"
            print(f"  [{gender}/{split_name}] {idx+1}/{len(split_saved)} ❌ {e}")

        results.append({
            "idx": idx,
            "text": gen_text,
            "gt_path": out_gt,
            "gen_path": out_gen,
            "status": status,
        })

    return results

print("=== Generating MALE train split ===")
male_train_results   = generate_for_split(male_train_saved,   MALE_REF_WAV,   MALE_REF_TEXT,   "male",   "train")

print("\n=== Generating MALE test split (UNSEEN) ===")
male_test_results    = generate_for_split(male_test_saved,    MALE_REF_WAV,   MALE_REF_TEXT,   "male",   "test")

print("\n=== Generating FEMALE train split ===")
female_train_results = generate_for_split(female_train_saved, FEMALE_REF_WAV, FEMALE_REF_TEXT, "female", "train")

print("\n=== Generating FEMALE test split (UNSEEN) ===")
female_test_results  = generate_for_split(female_test_saved,  FEMALE_REF_WAV, FEMALE_REF_TEXT, "female", "test")

# Save full results manifest
full_results = {
    "male_train":   male_train_results,
    "male_test":    male_test_results,
    "female_train": female_train_results,
    "female_test":  female_test_results,
}
with open(os.path.join(OUTPUT_ROOT, "split_results.json"), "w", encoding="utf-8") as f:
    json.dump(full_results, f, ensure_ascii=False, indent=2)

print("\n✅ All train/test generation done. Results in /content/outputs/")

=== Generating MALE train split ===
Converting audio...
Using custom reference text...

ref_text   भारत के बहोत ही बड़े इंडस्ट्रायल मुकेश अंबानी इनका जिओ का सिम है. 
gen_text 0 सैलरी से ख़ुशी मिलती है फिर हम कहीं टूर्नामेंट खेलने गए हैं हम बैटिंग करते हैं तो हम टूर्नामेंट एक बार खेलने गए थे तो वहाँ पर मैन ऑफ द सीरीज़ टी वी रखा हुआ था


Generating audio in 1 batches...


100%|██████████| 1/1 [00:31<00:00, 31.37s/it]


  [male/train] 1/10 ✅ gen saved
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   भारत के बहोत ही बड़े इंडस्ट्रायल मुकेश अंबानी इनका जिओ का सिम है. 
gen_text 0 फिर हम लोग गए हम देखो हम काम करते हैं डेली काम करते है डेली बेसिस पर तो डेली काम करते करते जब महिना पूरा होता है हमें सैलरी मिलती है तो सैलरी जब हमारे हाथ मे आता है तो हमें सैलरी से बहुत ख़ुशी मिलती है


Generating audio in 1 batches...


100%|██████████| 1/1 [00:40<00:00, 40.84s/it]


  [male/train] 2/10 ✅ gen saved
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   भारत के बहोत ही बड़े इंडस्ट्रायल मुकेश अंबानी इनका जिओ का सिम है. 
gen_text 0 भारत के बहोत ही बड़े इंडस्ट्रायल मुकेश अंबानी इनका जिओ का सिम है


Generating audio in 1 batches...


...

  [female/test] 3/5 ✅ gen saved
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   करेंगे कि ऐसे नहीं ऐसे जाओ यहाँ से यहाँ पर इलाज नहीं हो पाएगा इमरजेंसी है. 
gen_text 0 हमारे जीवन में बहुत आवश्यक है इस लिए स्वास्थ्य जो केंद्र हैं


Generating audio in 1 batches...


100%|██████████| 1/1 [00:12<00:00, 12.34s/it]


  [female/test] 4/5 ✅ gen saved
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   करेंगे कि ऐसे नहीं ऐसे जाओ यहाँ से यहाँ पर इलाज नहीं हो पाएगा इमरजेंसी है. 
gen_text 0 तो इलाहाबाद बनारस ले कर जाओ ऐसे नहीं करना चाहिए यहाँ हमें सभी सुविधाएं उपलब्ध करानी चाहिए अपने स्वास्थ्य केंद्र में ही सभी सुविधा उपलब्ध करना चाहिए और जो है नर्स वग़ैरह है वो अच्छे से देख भाल नहीं करती हैं


Generating audio in 1 batches...


100%|██████████| 1/1 [00:30<00:00, 30.40s/it]

  [female/test] 5/5 ✅ gen saved

✅ All train/test generation done. Results in /content/outputs/


In [9]:
import json

EVAL_SET_PATH = "/content/hindi_evaluation_set.json"

with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Top-level type : {type(raw)}")

if isinstance(raw, dict):
    print(f"Top-level keys : {list(raw.keys())}")
    for k, v in raw.items():
        print(f"  key='{k}'  type={type(v).__name__}  ", end="")
        if isinstance(v, list):
            print(f"len={len(v)}  first_item={v[0]}")
        elif isinstance(v, dict):
            print(f"keys={list(v.keys())}")
        else:
            print(f"value={str(v)[:80]}")
elif isinstance(raw, list):
    print(f"List length    : {len(raw)}")
    print(f"First item     : {raw[0]}")

Top-level type : <class 'dict'>
Top-level keys : ['vowels_and_consonants', 'velars_gutturals', 'retroflexes', 'palatals_and_nasals', 'labials_and_aspirated', 'loan_words_nukta', 'complex_conjuncts', 'dentals_and_visarga', 'flaps_and_chandrabindu', 'approximants_and_sibilants', 'english_loan_words', 'heavy_geminates', 'ha_placement', 'vowel_hiatus', 'sanskrit_tatsama', 'prosody_and_punctuation', 'perso_arabic_nukta', 'number_normalization', 'consonant_clusters_r', 'alliteration_rapid']
  key='vowels_and_consonants'  type=dict  keys=['id', 'text']
  key='velars_gutturals'  type=dict  keys=['id', 'text']
  key='retroflexes'  type=dict  keys=['id', 'text']
  key='palatals_and_nasals'  type=dict  keys=['id', 'text']
  key='labials_and_aspirated'  type=dict  keys=['id', 'text']
  key='loan_words_nukta'  type=dict  keys=['id', 'text']
  key='complex_conjuncts'  type=dict  keys=['id', 'text']
  key='dentals_and_visarga'  type=dict  keys=['id', 'text']
  key='flaps_and_chandrabindu'  type=dict 

In [10]:
import json

EVAL_SET_PATH = "/content/hindi_evaluation_set.json"

with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Structure: {"category_name": {"id": ..., "text": "..."}, ...}
# Extract all 20 entries preserving category name for reference
eval_set = []
for category, item in raw.items():
    eval_set.append({
        "id":       item["id"],
        "text":     item["text"],
        "category": category,
    })

TEXT_KEY = "text"

print(f"✅ Loaded {len(eval_set)} evaluation sentences.\n")
for i, item in enumerate(eval_set):
    print(f"  [{i+1:02d}] [{item['category']}]")
    print(f"       {item['text'][:90]}")

✅ Loaded 20 evaluation sentences.

  [01] [vowels_and_consonants]
       एक आम आदमी और औरत इमली के पेड़ के नीचे बैठकर ऊन बुन रहे हैं।
  [02] [velars_gutturals]
       कमल और काव्या ने खेत से कद्दू उखाड़ा, फिर गरम घी का घड़ा उठाकर घर की ओर भागे।
  [03] [retroflexes]
       षट्कोण के अंदर रखे ढक्कन और डमरू को देखकर ठग टमाटर टोकरी में डालकर डर गया।
  [04] [palatals_and_nasals]
       चंचल छतरी लेकर झमाझम बारिश में जंगल की ओर चली गई।
  [05] [labials_and_aspirated]
       भालू ने भारी पेड़ पर बैठकर मीठा फल खाया और पानी पी लिया।
  [06] [loan_words_nukta]
       ज़रा फ़िक्र मत करो, क़लम से ख़त लिखकर ग़ज़ल का मज़ा लो।
  [07] [complex_conjuncts]
       ज्ञानी ऋषि और श्रमिक ने क्षमा, विज्ञान और त्रिशूल का महत्व समझाया।
  [08] [dentals_and_visarga]
       दुःख मत कर, स्वतः ही नया धन प्राप्त होगा और धर्म की जीत होगी।
  [09] [flaps_and_chandrabindu]
       गाँव में पाँच ऊँट, एक साँड़, और बड़ी सूँड वाले बूढ़े हाथी खड़े थे।
  [10] [approximants_and_sibilants]
       यश और श्वेता बहुत विश्वास के साथ व

In [11]:
# ============================================================
# CELL 8: Generate all 20 eval sentences for MALE and FEMALE.
# OUTPUT:
#   /content/outputs/eval_male/eval_001.wav ...
#   /content/outputs/eval_female/eval_001.wav ...
# ============================================================

import os, json

EVAL_MALE_DIR   = "/content/outputs/eval_male"
EVAL_FEMALE_DIR = "/content/outputs/eval_female"
os.makedirs(EVAL_MALE_DIR,   exist_ok=True)
os.makedirs(EVAL_FEMALE_DIR, exist_ok=True)

eval_sentences = [item[TEXT_KEY] for item in eval_set[:20]]

def generate_eval(sentences, ref_wav, ref_text, out_dir, gender_label):
    results = []
    print(f"\n=== Generating EVAL speeches — {gender_label.upper()} ===")
    for idx, text in enumerate(sentences):
        out_path = os.path.join(out_dir, f"eval_{idx+1:03d}.wav")
        try:
            wav, sr, _ = tts_model.infer(
                ref_file     = ref_wav,
                ref_text     = ref_text,
                gen_text     = text,
                tokenizer    = "cls",
                cls_language = "hindi",
                file_wave    = out_path,
            )
            status = "ok"
            print(f"  [{idx+1}/20] ✅  {out_path}")
        except Exception as e:
            status = f"error: {e}"
            print(f"  [{idx+1}/20] ❌  {e}")

        results.append({
            "idx": idx + 1,
            "text": text,
            "wav_path": out_path,
            "status": status,
        })
    return results

male_eval_results   = generate_eval(eval_sentences, MALE_REF_WAV,   MALE_REF_TEXT,   EVAL_MALE_DIR,   "male")
female_eval_results = generate_eval(eval_sentences, FEMALE_REF_WAV, FEMALE_REF_TEXT, EVAL_FEMALE_DIR, "female")

# Save eval manifest
eval_manifest = {
    "male_speaker_id":   male_ref["speaker_id"],
    "female_speaker_id": female_ref["speaker_id"],
    "male_ref_wav":      MALE_REF_WAV,
    "female_ref_wav":    FEMALE_REF_WAV,
    "male_eval":   male_eval_results,
    "female_eval": female_eval_results,
}
with open("/content/outputs/eval_manifest.json", "w", encoding="utf-8") as f:
    json.dump(eval_manifest, f, ensure_ascii=False, indent=2)

print("\n✅ All 20 evaluation speeches generated for both male and female.")
print(f"   Male   → {EVAL_MALE_DIR}")
print(f"   Female → {EVAL_FEMALE_DIR}")


=== Generating EVAL speeches — MALE ===
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   भारत के बहोत ही बड़े इंडस्ट्रायल मुकेश अंबानी इनका जिओ का सिम है. 
gen_text 0 एक आम आदमी और औरत इमली के पेड़ के नीचे बैठकर ऊन बुन रहे हैं।


Generating audio in 1 batches...


100%|██████████| 1/1 [00:17<00:00, 18.00s/it]


  [1/20] ✅  /content/outputs/eval_male/eval_001.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   भारत के बहोत ही बड़े इंडस्ट्रायल मुकेश अंबानी इनका जिओ का सिम है. 
gen_text 0 कमल और काव्या ने खेत से कद्दू उखाड़ा, फिर गरम घी का घड़ा उठाकर घर की ओर भागे।


Generating audio in 1 batches...


100%|██████████| 1/1 [00:20<00:00, 20.49s/it]


  [2/20] ✅  /content/outputs/eval_male/eval_002.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   भारत के बहोत ही बड़े इंडस्ट्रायल मुकेश अंबानी इनका जिओ का सिम है. 
gen_text 0 षट्कोण के अंदर रखे ढक्कन और डमरू को देखकर ठग टमाटर टोकरी में डालकर डर गया।


Generating audio in 1 batches...


...

  [18/20] ✅  /content/outputs/eval_female/eval_018.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   करेंगे कि ऐसे नहीं ऐसे जाओ यहाँ से यहाँ पर इलाज नहीं हो पाएगा इमरजेंसी है. 
gen_text 0 ट्रेन स्टेशन से प्रस्थान कर चुकी है, कृपया अपने ट्रक को क्रॉसिंग से दूर रखें।


Generating audio in 1 batches...


100%|██████████| 1/1 [00:14<00:00, 14.47s/it]


  [19/20] ✅  /content/outputs/eval_female/eval_019.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   करेंगे कि ऐसे नहीं ऐसे जाओ यहाँ से यहाँ पर इलाज नहीं हो पाएगा इमरजेंसी है. 
gen_text 0 चंदू के चाचा ने चाँदी के चम्मच से चटपटी चटनी चखाई और चंपारण चले गए।


Generating audio in 1 batches...


100%|██████████| 1/1 [00:13<00:00, 13.24s/it]

  [20/20] ✅  /content/outputs/eval_female/eval_020.wav

✅ All 20 evaluation speeches generated for both male and female.
   Male   → /content/outputs/eval_male
   Female → /content/outputs/eval_female


In [12]:
# ============================================================
# CELL 9: Zip the entire outputs folder and download it.
# ============================================================

import shutil
from google.colab import files

zip_path = "/content/sooktam2_outputs"
print("Zipping /content/outputs/ ...")
shutil.make_archive(zip_path, "zip", "/content/outputs")
print(f"✅ Archive created: {zip_path}.zip")

print("Starting download...")
files.download(f"{zip_path}.zip")

Zipping /content/outputs/ ...
✅ Archive created: /content/sooktam2_outputs.zip
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>